In [ ]:
import mne
import numpy as np
import pandas as pd
import gc
import torch
import torch.optim as optim
import torch.nn as nn
from sklearn.metrics import accuracy_score
import time
from DeepSleep import DeepSleep
from utils import calculate_advanced_metrics, plot_confusion_matrix
# --- Function to extract sleep data (Sleep-EDF) ---
def load_sleep_data(subject_id):
    # Download data for the subject (use recording 1)
    files = mne.datasets.sleep_physionet.age.fetch_data(subjects=[subject_id], recording=[1])[0]
    raw = mne.io.read_raw_edf(files[0], preload=True)
    annot = mne.read_annotations(files[1])
    raw.set_annotations(annot, emit_warning=False)
    
    # I only select EEG channels (this avoids carrying EOG, EMG or other 5 useless channels) 
    raw.pick_channels(['EEG Fpz-Cz', 'EEG Pz-Oz'])
    raw.resample(100.0) # the DeepSleep paper ssumes 100Hz
    
    # Mapping sleep stages into 5 classes
    mapping = {'Sleep stage W': 1, 'Sleep stage 1': 2, 'Sleep stage 2': 3,
               'Sleep stage 3': 4, 'Sleep stage 4': 4, 'Sleep stage R': 5}
    
    # events every 30 seconds
    events, _ = mne.events_from_annotations(raw, event_id=mapping, chunk_duration=30.)
    
    # cut the data into 30-second epochs (3000 samples at 100Hz)
    tmax = 30. - 1. / raw.info['sfreq']
    epochs = mne.Epochs(raw=raw, events=events, event_id=[1, 2, 3, 4, 5],
                        tmin=0., tmax=tmax, baseline=None, preload=True)
    
    X = epochs.get_data(copy=True) * 1e6 # convert from Volts to microVolts (the DeepSleep paper uses microVolts)
    y = epochs.events[:, 2] - 1 # scale classes from 0 to 4
    
    return X, y

print("Libraries and functions loaded successfully")

Libraries and functions loaded successfully!


In [ ]:
import os

num_cores = os.cpu_count()
torch.set_num_threads(num_cores)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"System: {device} | Available CPU cores: {num_cores}")

Sistema: cpu | Core CPU pronti: 6


In [ ]:
lista_soggetti = list(range(5)) # PhysioNet Sleep-EDF uses indices starting from 0.
X_list, y_list, meta_list = [], [], []

print("Sleep-EDF data extraction in progress (may take about a minute for download)...")
for sbj in lista_soggetti:
    print(f"Processing subject {sbj}...")
    X_tmp, y_tmp = load_sleep_data(sbj)
    
    X_list.append(X_tmp.astype('float32'))
    y_list.append(y_tmp)
    # create metadata manually since we're not using MOABB
    meta_list.append(pd.DataFrame({'subject': [sbj] * len(y_tmp)}))
    
    del X_tmp
    gc.collect()

X = np.concatenate(X_list, axis=0)
y = np.concatenate(y_list, axis=0)
metadata = pd.concat(meta_list, ignore_index=True)

print(f"\n Data extraction completed")
print(f"Shape X: {X.shape} -> (trials, 2 channels, 3000 samples)")

Estrazione dati Sleep-EDF in corso (potrebbe richiedere un minuto per il download)...
Elaborazione Soggetto 0...
Extracting EDF parameters from /home/devcontainers/mne_data/physionet-sleep-data/SC4001E0-PSG.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 7949999  =      0.000 ... 79499.990 secs...


/home/devcontainers/miniconda3/envs/eeg_clean/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_69308/2848585093.py:14: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(files[0], preload=True)
/tmp/ipykernel_69308/2848585093.py:14: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(files[0], preload=True)
/tmp/ipykernel_69308/2848585093.py:14: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(files[0], preload=True)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Sampling frequency of the instance is already 100.0, returning unmodified.
Used Annotations descriptions: [np.str_('Sleep stage 1'), np.str_('Sleep stage 2'), np.str_('Sleep stage 3'), np.str_('Sleep stage 4'), np.str_('Sleep stage R'), np.str_('Sleep stage W')]
Not setting metadata
2650 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 2650 events and 3000 original time points ...
0 bad epochs dropped
Elaborazione Soggetto 1...
Extracting EDF parameters from /home/devcontainers/mne_data/physionet-sleep-data/SC4011E0-PSG.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 8405999  =      0.000 ... 84059.990 secs...


/tmp/ipykernel_69308/2848585093.py:14: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(files[0], preload=True)
/tmp/ipykernel_69308/2848585093.py:14: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(files[0], preload=True)
/tmp/ipykernel_69308/2848585093.py:14: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(files[0], preload=True)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Sampling frequency of the instance is already 100.0, returning unmodified.
Used Annotations descriptions: [np.str_('Sleep stage 1'), np.str_('Sleep stage 2'), np.str_('Sleep stage 3'), np.str_('Sleep stage 4'), np.str_('Sleep stage R'), np.str_('Sleep stage W')]
Not setting metadata
2802 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 2802 events and 3000 original time points ...
0 bad epochs dropped
Elaborazione Soggetto 2...
Extracting EDF parameters from /home/devcontainers/mne_data/physionet-sleep-data/SC4021E0-PSG.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 8411999  =      0.000 ... 84119.990 secs...


...

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Sampling frequency of the instance is already 100.0, returning unmodified.
Used Annotations descriptions: [np.str_('Sleep stage 1'), np.str_('Sleep stage 2'), np.str_('Sleep stage 3'), np.str_('Sleep stage 4'), np.str_('Sleep stage R'), np.str_('Sleep stage W')]
Not setting metadata
2804 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 2804 events and 3000 original time points ...
0 bad epochs dropped
Elaborazione Soggetto 3...
Extracting EDF parameters from /home/devcontainers/mne_data/physionet-sleep-data/SC4031E0-PSG.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 8459999  =      0.000 ... 84599.990 secs...


/tmp/ipykernel_69308/2848585093.py:14: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(files[0], preload=True)
/tmp/ipykernel_69308/2848585093.py:14: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(files[0], preload=True)
/tmp/ipykernel_69308/2848585093.py:14: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(files[0], preload=True)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Sampling frequency of the instance is already 100.0, returning unmodified.
Used Annotations descriptions: [np.str_('Sleep stage 1'), np.str_('Sleep stage 2'), np.str_('Sleep stage 3'), np.str_('Sleep stage 4'), np.str_('Sleep stage R'), np.str_('Sleep stage W')]
Not setting metadata
2820 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 2820 events and 3000 original time points ...
0 bad epochs dropped
Elaborazione Soggetto 4...
Extracting EDF parameters from /home/devcontainers/mne_data/physionet-sleep-data/SC4041E0-PSG.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 7709999  =      0.000 ... 77099.990 secs...


/tmp/ipykernel_69308/2848585093.py:14: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(files[0], preload=True)
/tmp/ipykernel_69308/2848585093.py:14: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(files[0], preload=True)
/tmp/ipykernel_69308/2848585093.py:14: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(files[0], preload=True)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Sampling frequency of the instance is already 100.0, returning unmodified.
Used Annotations descriptions: [np.str_('Sleep stage 1'), np.str_('Sleep stage 2'), np.str_('Sleep stage 3'), np.str_('Sleep stage R'), np.str_('Sleep stage W')]
Not setting metadata
2569 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 2569 events and 3000 original time points ...
0 bad epochs dropped

✅ Caricamento completato!
Shape X: (13645, 2, 3000) -> (prove, 2 canali, 3000 campioni)


X Shape: (13645, 2, 3000) -> (trials, 2 channels, 3000 samples)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

X_tensor = torch.from_numpy(X).float()
y_tensor = torch.from_numpy(y).long()

n_channels = X.shape[1]
n_samples = X.shape[2]
n_classes = len(np.unique(y))

class_counts = np.bincount(y)
weights = 1.0 / torch.tensor(class_counts, dtype=torch.float32)
weights = weights / weights.sum() * n_classes
weights = weights.to(device)

all_y_true = []
all_y_pred = []

print("\n=== Start LOSO Validation on DeepSleep ===")
loso_results = []
subjects_array = metadata['subject'].unique()
num_epochs = 10 # 10 epochs are enough to see if the model learns something

for test_subject in subjects_array:
    print(f"\n-> Training for Test Subject: {test_subject}")
    
    train_mask = (metadata['subject'] != test_subject).values
    test_mask = (metadata['subject'] == test_subject).values
    
    # DeepSleep Initialization with Latent Alignment
    model = DeepSleep(in_shape=(n_channels, n_samples), n_out=n_classes, alignment='latent').to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    train_subjects = metadata[train_mask]['subject'].unique()
    
    model.train()
    start_time = time.time()
    
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for subj in train_subjects:
            subj_mask = (metadata['subject'] == subj).values
            batch_X = X_tensor[subj_mask].to(device)
            batch_y = y_tensor[subj_mask].to(device)
            
            optimizer.zero_grad(set_to_none=True)

            criterion = optim.Adam(model.parameters(), lr=0.001)
            
            # Pass the number of trials for this subject to the model for proper alignment
            outputs = model(batch_X, sbj_trials=batch_X.shape[0])
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            
        if (epoch + 1) % 5 == 0:
            print(f"   Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss/len(train_subjects):.4f}")
            
    # Assessment on the left-out subject
    model.eval()
    with torch.no_grad():
        test_X = X_tensor[test_mask].to(device)
        test_y = y_tensor[test_mask]
        outputs = model(test_X, sbj_trials=test_X.shape[0])
        _, predicted = torch.max(outputs.data, 1)
        
        y_true_np=test_y.cpu().numpy()
        y_pred_np=predicted.cpu().numpy()

        all_y_true.extend(y_true_np)
        all_y_pred.extend(y_pred_np)

        acc = accuracy_score(test_y.cpu().numpy(), predicted.cpu().numpy())
        loso_results.append(acc)
        
    print(f"-> End of Test Subject  {test_subject} | Accuracy: {acc*100:.2f}% | Time: {time.time()-start_time:.1f}s")

print(f"\nFINAL RESULTS LOSO DeepSleep (Latent Alignment): Mean {np.mean(loso_results)*100:.2f}%")

all_y_true = np.array(all_y_true)
all_y_pred = np.array(all_y_pred)

sleep_stages = ['W', 'N1', 'N2', 'N3', 'REM']
calculate_advanced_metrics(all_y_true, all_y_pred, class_names=sleep_stages)

plot_confusion_matrix(all_y_true, all_y_pred, class_names=sleep_stages, 
                                title="DeepSleep + Latent Alignment Confusion Matrix")

Utilizzando il device: cpu

=== Inizio Validazione LOSO su DeepSleep ===

-> Addestramento per Test Subject: 0
   Epoca [5/10], Loss: 0.5747
   Epoca [10/10], Loss: 0.4354
-> Fine Test Subject 0 | Accuracy: 83.58% | Tempo: 420.5s

-> Addestramento per Test Subject: 1
   Epoca [5/10], Loss: 0.5054
   Epoca [10/10], Loss: 0.3708
-> Fine Test Subject 1 | Accuracy: 86.01% | Tempo: 393.5s

-> Addestramento per Test Subject: 2
   Epoca [5/10], Loss: 0.5277
   Epoca [10/10], Loss: 0.3854
-> Fine Test Subject 2 | Accuracy: 89.09% | Tempo: 385.4s

-> Addestramento per Test Subject: 3
   Epoca [5/10], Loss: 0.6147
   Epoca [10/10], Loss: 0.4739
-> Fine Test Subject 3 | Accuracy: 87.20% | Tempo: 384.8s

-> Addestramento per Test Subject: 4
   Epoca [5/10], Loss: 0.5437
   Epoca [10/10], Loss: 0.3824
-> Fine Test Subject 4 | Accuracy: 82.68% | Tempo: 394.5s

RISULTATI FINALI LOSO DeepSleep (Latent Alignment): Media 85.71%


```text
 Using device: cuda

=== Start LOSO Validation on DeepSleep ===

-> Training for Test Subject: 0
   Epoch [5/10], Loss: 0.9438
   Epoch [10/10], Loss: 0.7068
-> End of Test Subject  0 | Accuracy: 87.13% | Time: 27.4s

-> Training for Test Subject: 1
   Epoch [5/10], Loss: 0.9552
   Epoch [10/10], Loss: 0.6930
-> End of Test Subject  1 | Accuracy: 87.44% | Time: 27.5s

-> Training for Test Subject: 2
   Epoch [5/10], Loss: 1.0136
   Epoch [10/10], Loss: 0.7809
-> End of Test Subject  2 | Accuracy: 84.09% | Time: 27.9s

-> Training for Test Subject: 3
   Epoch [5/10], Loss: 1.0794
   Epoch [10/10], Loss: 0.8170
-> End of Test Subject  3 | Accuracy: 84.72% | Time: 28.3s

-> Training for Test Subject: 4
   Epoch [5/10], Loss: 0.9075
   Epoch [10/10], Loss: 0.6575
-> End of Test Subject  4 | Accuracy: 79.06% | Time: 28.6s

FINAL RESULTS LOSO DeepSleep (Latent Alignment): Mean 84.49%

=======================================================
ADVANCED CLINICAL METRICS REPORT
=======================================================

COHEN'S KAPPA: 0.7091 (Good Agreement)

MACRO-AVERAGED SPECIFICITY: 96.64%

DETAILED REPORT (Scikit-Learn):
              precision    recall  f1-score   support

           W       1.00      0.93      0.96      9302
          N1       0.09      0.12      0.10       488
          N2       0.89      0.68      0.77      2462
          N3       0.56      0.88      0.69       530
         REM       0.42      0.81      0.56       863

    accuracy                           0.85     13645
   macro avg       0.59      0.68      0.62     13645
weighted avg       0.89      0.85      0.86     13645

=======================================================


![Confusion Matrix with Latent alignment](./confusion_matrix/confusion_matrix_deep_sleep_latent_alignment.png)

In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn
from sklearn.metrics import accuracy_score
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_tensor = torch.from_numpy(X).float()
y_tensor = torch.from_numpy(y).long()

n_channels = X.shape[1]
n_samples = X.shape[2]
n_classes = len(np.unique(y))

all_y_true_base = []
all_y_pred_base = []

print("\n=== Start of LOSO validation on DeepSleep (BASELINE) ===")
loso_results_baseline = []
subjects_array = metadata['subject'].unique()
num_epochs = 10 

for test_subject in subjects_array:
    print(f"\n-> Training for Test Subject: {test_subject}")
    
    train_mask = (metadata['subject'] != test_subject).values
    test_mask = (metadata['subject'] == test_subject).values

    # preparation of training data (all together, we don't iterate by subject)
    X_train_loso = X_tensor[train_mask].to(device)
    y_train_loso = y_tensor[train_mask].to(device)
    
    # DeepSleep Initialization without Latent Alignment
    model = DeepSleep(in_shape=(n_channels, n_samples), n_out=n_classes, alignment='None').to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    model.train()
    start_time = time.time()
    
    for epoch in range(num_epochs):
        optimizer.zero_grad(set_to_none=True)
        
        # pass the entire block at once (in the baseline sbj_trials doesn't matter)
        outputs = model(X_train_loso, sbj_trials=X_train_loso.shape[0])
        loss = criterion(outputs, y_train_loso)
        loss.backward()
        optimizer.step()
        
        if (epoch + 1) % 5 == 0:
            print(f"   Epoca [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")
            
    # Valutation
    model.eval()
    with torch.no_grad():
        test_X = X_tensor[test_mask].to(device)
        test_y = y_tensor[test_mask] # remains in CPU for accuracy_score
        outputs = model(test_X, sbj_trials=test_X.shape[0])
        _, predicted = torch.max(outputs.data, 1)
        
        y_true_np=test_y.cpu().numpy()
        y_pred_np=predicted.cpu().numpy()

        all_y_true_base.extend(y_true_np)
        all_y_pred_base.extend(y_pred_np)

        acc = accuracy_score(test_y.cpu().numpy(), predicted.cpu().numpy())
        loso_results_baseline.append(acc)

    # Clean up to free GPU memory
    del X_train_loso, y_train_loso, test_X, model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"-> End of Test Subject  {test_subject} | Accuracy: {acc*100:.2f}% | Time: {time.time()-start_time:.1f}s")

print("\n" + "="*50)
print(f"FINAL RESULTS LOSO DeepSleep (Baseline)")
print(f"Mean Accuracy: {np.mean(loso_results_baseline)*100:.2f}% ± {np.std(loso_results_baseline)*100:.2f}%")
print("="*50)

all_y_true_base = np.array(all_y_true_base)
all_y_pred_base = np.array(all_y_pred_base)

print("\n--- ANALISI MODELLO BASELINE ---")
calculate_advanced_metrics(all_y_true_base, all_y_pred_base, class_names=sleep_stages)

=== Starting LOSO Validation on DeepSleep (BASELINE) ===
```text
=== Start of LOSO validation on DeepSleep (BASELINE) ===

-> Training for Test Subject: 0
   Epoca [5/10], Loss: 1.2559
   Epoca [10/10], Loss: 1.0735
-> End of Test Subject  0 | Accuracy: 78.98% | Time: 21.4s

-> Training for Test Subject: 1
   Epoca [5/10], Loss: 1.4150
   Epoca [10/10], Loss: 1.2275
-> End of Test Subject  1 | Accuracy: 65.06% | Time: 21.3s

-> Training for Test Subject: 2
   Epoca [5/10], Loss: 1.3143
   Epoca [10/10], Loss: 1.0900
-> End of Test Subject  2 | Accuracy: 53.07% | Time: 21.5s

-> Training for Test Subject: 3
   Epoca [5/10], Loss: 1.2331
   Epoca [10/10], Loss: 1.0530
-> End of Test Subject  3 | Accuracy: 59.01% | Time: 21.7s

-> Training for Test Subject: 4
   Epoca [5/10], Loss: 1.3410
   Epoca [10/10], Loss: 1.2001
-> End of Test Subject  4 | Accuracy: 62.20% | Time: 22.3s

==================================================
FINAL RESULTS LOSO DeepSleep (Baseline)
Mean Accuracy: 63.66% ± 8.63%
==================================================

--- ANALISI MODELLO BASELINE ---

=======================================================
ADVANCED CLINICAL METRICS REPORT
=======================================================

COHEN'S KAPPA: 0.3658 (Fair Agreement)

MACRO-AVERAGED SPECIFICITY: 91.30%

DETAILED REPORT (Scikit-Learn):
              precision    recall  f1-score   support

           W       0.96      0.82      0.89      9302
          N1       0.24      0.02      0.04       488
          N2       0.57      0.20      0.29      2462
          N3       0.09      0.69      0.15       530
         REM       0.27      0.17      0.21       863

    accuracy                           0.64     13645
   macro avg       0.43      0.38      0.32     13645
weighted avg       0.79      0.64      0.68     13645

=======================================================


![Confusion Matrix with Latent alignment](./confusion_matrix/confusion_matrix_deep_sleep_baseline.png)